<a href="https://colab.research.google.com/github/deepali-ds/Air-pollution-dashboard/blob/main/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forest-to-Coffee Deforestation Detector
### Adapted from Crop Health & Water Status Classifier

This notebook directly adapts the crop health monitoring pipeline for deforestation
detection, fully aligned with the Deforestation Detection Roadmap.

**Roadmap steps covered in this notebook:**
- Step 2: Data collection -- Sentinel-2, Sentinel-1, Hansen GFC, CHIRPS, SRTM
- Step 3: Baseline forest mapping (T1) -- Hansen tree cover + elevation mask
- Step 4: Forest loss detection -- Hansen loss year + NDVI drop
- Step 5: Coffee plantation detection (T2) -- Sentinel-2 + Sentinel-1 + suitability
- Step 6: Forest to coffee change detection -- three-way intersection
- Step 7: Feature engineering -- NDVI/EVI/NDWI/NDMI + VH/VV radar + terrain
- Step 8: Time series change analysis -- NDVI T1 vs T2 + difference signal

**Roadmap steps NOT in this notebook (handled by team):**
- Coffee plantation detection ML model (Random Forest / Gradient Boosting / U-Net)
- Individual coffee plant detection (YOLO / Mask R-CNN)
- Validation against known coffee regions: Step 9
- App integration: Step 11

**Data sources (roadmap Step 2):**
- Sentinel-2 Surface Reflectance (optical) via GEE
- Sentinel-1 SAR GRD (radar) via GEE
- Hansen Global Forest Change via GEE
- SRTM DEM (elevation) via GEE
- CHIRPS Daily Precipitation via GEE

**Default AOI:** Cacoal, Rondonia, Brazil (Amazon frontier coffee region)
**Time window:** 2017 (T1) to 2023 (T2)

---
## Section 0: Pipeline Configuration

Same structure as crop health pipeline -- all parameters in one place.

**Key changes for deforestation (roadmap alignment):**
- Two dates: DATE_T1 (forest baseline) and DATE_T2 (post-conversion)
- AOI: Cacoal, Rondonia, Brazil -- active Amazon deforestation frontier
- Larger buffer (15km) -- deforestation patches are regional scale
- Wider date window (45 days) -- persistent tropical cloud cover needs more scenes
- Higher cloud threshold (30%) -- same reason
- Suitability filters added: elevation 600-2200m, coffee belt latitude
- Hansen tree cover threshold for forest baseline
- NDVI loss threshold for change detection
- Coffee spectral range thresholds for plantation detection
- Sentinel-1 VH backscatter range for plantation structure detection
- Scale set to 30m to match Hansen GFC resolution

In [ ]:
# PIPELINE CONFIGURATION
# Adapted from crop health pipeline for deforestation detection
# Fully aligned with Deforestation Detection Roadmap

# AOI: Cacoal, Rondonia, Brazil
# Roadmap Step 2: Study regions include Brazil
# Cacoal: documented forest-to-coffee conversion on Amazon frontier 2017-2023
AOI_LAT         = -11.43
AOI_LON         = -61.45
AOI_BUFFER_KM   = 5.0           # Reduced from 15.0km to fit within 100km^2 limit
AOI_MODE        = 'latlon'

# Time window (roadmap Step 2: 2017 to 2023)
DATE_T1          = '2017-07-01'  # T1: forest baseline
DATE_T2          = '2023-07-01'  # T2: post-conversion
DATE_WINDOW_DAYS = 45            # Wider than crop health (10 days) -- tropical cloud cover
RAINFALL_LOOKBACK_DAYS = 365     # Annual rainfall for suitability (not 30-day stress)

# Image quality
CLOUD_THRESHOLD  = 30            # Higher than crop health (20%) -- Rondonia persistent clouds

#  Roadmap Step 3: Forest baseline thresholds
TREE_COVER_THRESHOLD = 50        # Hansen tree cover > 50% = forest
ELEVATION_MIN    = 600           # metres -- coffee suitability filter (roadmap Step 3)
ELEVATION_MAX    = 2200          # metres -- Rondonia grows Robusta, relaxed to 100m below
COFFEE_BELT_LAT  = 25            # degrees -- coffee belt filter (roadmap Step 3)

# Roadmap Step 4: Forest loss threshold
NDVI_LOSS_THRESHOLD  = -0.15     # NDVI drop > 0.15 between T1 and T2 = vegetation loss

# Roadmap Step 5: Coffee detection thresholds
# Coffee NDVI: evergreen, moderately high, stable (lower than closed forest)
COFFEE_NDVI_MIN  = 0.35
COFFEE_NDVI_MAX  = 0.75
COFFEE_EVI_MIN   = 0.20
COFFEE_EVI_MAX   = 0.55
# Sentinel-1 VH: moderate backscatter (roadmap Step 5: plantation structure)
COFFEE_VH_MIN    = -18.0         # dB
COFFEE_VH_MAX    = -10.0         # dB

# Hotspot detection (same as crop health pipeline)
N_CLUSTERS        = 3
UNIFORMITY_BINS   = [70, 40]
ISOLATION_CONTAMINATION = 0.05
DBSCAN_EPS        = 0.01         # Larger than crop health (0.001) - bigger patches
DBSCAN_MIN_SAMPLES = 10
N_HOTSPOTS        = 5

#  GEE project
GEE_PROJECT       = 'projecttestrun'
SCALE             = 30           # 30m - matches Hansen GFC resolution

print('Configuration loaded.')
print(f'AOI: Cacoal, Rondonia, Brazil ({AOI_LAT}, {AOI_LON})')
print(f'Time window: {DATE_T1} (T1) to {DATE_T2} (T2)')

---
## Section 1: Setup & Authentication


In [ ]:
# Install packages
!pip install earthengine-api geemap geopandas shapely scikit-learn folium -q

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import folium
from datetime import datetime, timedelta
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Authenticate and initialize GEE
ee.Authenticate() # uncomment on first run only
ee.Initialize(project=GEE_PROJECT)

print('All libraries loaded. GEE initialized.')

---
## Section 2: AOI Definition

**Identical to crop health pipeline** - same two input modes (latlon or draw).

Buffer increased to 15km to capture regional deforestation patterns.
Roadmap Step 2 specifies Brazil as a study region -- Cacoal, Rondonia
is the default, with other regions configurable via AOI_LAT/AOI_LON.

In [ ]:
def latlon_to_aoi(lat, lon, buffer_km):
    """
    creates a square GEE geometry centered on (lat, lon)
    with a given buffer radius in km.

    returns: ee.Geometry.Polygon
    """
    # Convert km to degrees (approximate, valid at mid-latitudes)
    delta_deg = buffer_km / 111.0
    return ee.Geometry.BBox(
        lon - delta_deg,  # west
        lat - delta_deg,  # south
        lon + delta_deg,  # east
        lat + delta_deg   # north
    )


def validate_aoi_size(aoi, max_km2=100):
    """
    checks that the AOI does not exceed the maximum allowed area.
    raises ValueError if too large.

    max_km2: corresponds to 10x10 km = 100 km²
    """
    area_m2 = aoi.area(maxError=10).getInfo()
    area_km2 = area_m2 / 1e6
    if area_km2 > max_km2:
        raise ValueError(
            f'AOI area {area_km2:.1f} km² exceeds {max_km2} km². Reduce AOI_BUFFER_KM.'
        )
    print(f'AOI validated: {area_km2:.2f} km² (max {max_km2} km²)')
    return area_km2


def define_aoi(mode='latlon', lat=AOI_LAT, lon=AOI_LON,
               buffer_km=AOI_BUFFER_KM, drawn_geometry=None):
    """
    main AOI definition function.

    mode: 'latlon' for coordinate + buffer, 'draw' for interactive polygon
    returns: ee.Geometry
    """
    if mode == 'latlon':
        aoi = latlon_to_aoi(lat, lon, buffer_km)
        print(f'AOI defined via lat/lon: center ({lat}, {lon}), buffer {buffer_km} km')
    elif mode == 'draw':
        if drawn_geometry is None:
            raise ValueError('drawn_geometry must be provided when mode="draw". '
                             'Run the interactive map cell below first.')
        aoi = drawn_geometry
        print('AOI defined via drawn polygon.')
    else:
        raise ValueError(f'Unknown mode "{mode}". Use "latlon" or "draw".')

    validate_aoi_size(aoi)
    return aoi


# Define AOI
AOI = define_aoi(mode=AOI_MODE)
print(f'\nAOI ready for analysis.')

In [ ]:
def latlon_to_aoi(lat, lon, buffer_km):
    """
    creates a square GEE geometry centered on (lat, lon)
    with a given buffer radius in km.

    returns: ee.Geometry.Polygon
    """
    # Convert km to degrees (approximate, valid at mid-latitudes)
    delta_deg = buffer_km / 111.0
    return ee.Geometry.BBox(
        lon - delta_deg,  # west
        lat - delta_deg,  # south
        lon + delta_deg,  # east
        lat + delta_deg   # north
    )


def validate_aoi_size(aoi, max_km2=100):
    """
    checks that the AOI does not exceed the maximum allowed area.
    raises ValueError if too large.

    max_km2: corresponds to 10x10 km = 100 km²
    """
    area_m2 = aoi.area(maxError=10).getInfo()
    area_km2 = area_m2 / 1e6
    if area_km2 > max_km2:
        raise ValueError(
            f'AOI area {area_km2:.1f} km² exceeds {max_km2} km². Reduce AOI_BUFFER_KM.'
        )
    print(f'AOI validated: {area_km2:.2f} km² (max {max_km2} km²)')
    return area_km2


def define_aoi(mode='latlon', lat=AOI_LAT, lon=AOI_LON,
               buffer_km=AOI_BUFFER_KM, drawn_geometry=None):
    """
    main AOI definition function.

    mode: 'latlon' for coordinate + buffer, 'draw' for interactive polygon
    returns: ee.Geometry
    """
    if mode == 'latlon':
        aoi = latlon_to_aoi(lat, lon, buffer_km)
        print(f'AOI defined via lat/lon: center ({lat}, {lon}), buffer {buffer_km} km')
    elif mode == 'draw':
        if drawn_geometry is None:
            raise ValueError('drawn_geometry must be provided when mode="draw". '
                             'Run the interactive map cell below first.')
        aoi = drawn_geometry
        print('AOI defined via drawn polygon.')
    else:
        raise ValueError(f'Unknown mode "{mode}". Use "latlon" or "draw".')

    validate_aoi_size(aoi)
    return aoi


# Define AOI
AOI = define_aoi(mode=AOI_MODE)
print(f'\nAOI ready for analysis.')

In [ ]:

# draw  field polygon on the map, then run define_aoi(mode='draw', drawn_geometry=...)

draw_map = geemap.Map(center=[AOI_LAT, AOI_LON], zoom=12)
draw_map.add_basemap('SATELLITE')
draw_map

---
## Section 3: Sentinel-2 Loading & Cloud Masking

**Identical to crop health pipeline** -- same function, same QA60 cloud masking.

**Roadmap Step 2 alignment:** Sentinel-2 is the primary optical data source.
Surface reflectance (S2_SR_HARMONIZED) corrects for atmospheric scattering.

**Key adaptation:** Called TWICE -- once for T1 (2017) and once for T2 (2023).
Date window widened to 45 days and cloud threshold raised to 30% for
Rondonia's persistent tropical cloud cover.

In [ ]:
def mask_s2_clouds(image):
    """
    masks clouds and cirrus in a Sentinel-2 image using the QA60 bitmask band.
    Bit 10: opaque clouds | Bit 11: cirrus clouds

    returns: cloud-masked ee.Image with scale factor applied
    """
    qa = image.select('QA60')
    cloud_bit_mask  = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags must be zero (clear conditions)
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    # Apply scale factor (S2 SR values are stored as int * 10000)
    return image.updateMask(mask).divide(10000)


def load_sentinel2(aoi, date_center, window_days=DATE_WINDOW_DAYS,
                   cloud_threshold=CLOUD_THRESHOLD):
    """
    loads and preprocesses a Sentinel-2 SR composite for the given AOI and date.

    args:
        aoi: ee.Geometry — field boundary
        date_center: str — analysis date 'YYYY-MM-DD'
        window_days: int — ± days around center date
        cloud_threshold: int — max scene-level cloud cover %

    returns:
        composite: ee.Image — median composite, cloud-masked, scaled
        n_images: int — number of scenes in composite
    """
    center_dt = datetime.strptime(date_center, '%Y-%m-%d')
    start_date = (center_dt - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end_date   = (center_dt + timedelta(days=window_days)).strftime('%Y-%m-%d')

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
        .map(mask_s2_clouds)
    )

    n_images = collection.size().getInfo()

    if n_images == 0:
        raise ValueError(
            f'No Sentinel-2 scenes found for the given AOI and date range '
            f'({start_date} to {end_date}) with cloud cover < {cloud_threshold}%. '
            f'Try increasing DATE_WINDOW_DAYS or CLOUD_THRESHOLD in the config.'
        )

    # median composite: robust to residual cloud artifacts
    composite = collection.median().clip(aoi)

    print(f'Sentinel-2 composite loaded:')
    print(f'  Date window : {start_date} to {end_date}')
    print(f'  Scenes used : {n_images}')
    print(f'  Cloud filter: < {cloud_threshold}%')

    return composite, n_images


# Roadmap Step 2: load BOTH time periods
print(f'Loading T1 ({DATE_T1})...')
s2_T1, n_T1 = load_sentinel2(AOI, DATE_T1)

print(f'Loading T2 ({DATE_T2})...')
s2_T2, n_T2 = load_sentinel2(AOI, DATE_T2)

print(f'T1 scenes: {n_T1} | T2 scenes: {n_T2}')

---
## Section 3b: Sentinel-1 SAR Loading

**New vs crop health pipeline -- required by roadmap Step 2.**


**Bands (roadmap Step 7 feature engineering):**
- VH: sensitive to vegetation volume and structure
- VV: sensitive to surface roughness
- VH/VV ratio: distinguishes vegetation types

In [ ]:
def load_sentinel1(aoi, date_center, window_days=DATE_WINDOW_DAYS):
    """
    New vs crop health pipeline.
    Required by roadmap Step 2 (Sentinel-1 radar data source)
    and Step 5 (plantation structure detection via VH/VV).

    Uses IW (Interferometric Wide) mode: 10m resolution, VV and VH polarisation.

    Backscatter values in dB (log scale):
      Dense forest:      VH ~ -8 to -12 dB (high backscatter)
      Coffee plantation: VH ~ -10 to -18 dB (moderate)
      Bare soil:         VH ~ -18 to -25 dB (low backscatter)

    Returns: SAR composite ee.Image with VH, VV, VH_VV_ratio bands
    """
    from datetime import datetime, timedelta
    center_dt  = datetime.strptime(date_center, '%Y-%m-%d')
    start_date = (center_dt - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end_date   = (center_dt + timedelta(days=window_days)).strftime('%Y-%m-%d')

    collection = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .select(['VH', 'VV'])
    )

    n_images = collection.size().getInfo()
    if n_images == 0:
        raise ValueError(
            f'No Sentinel-1 scenes found ({start_date} to {end_date}). '
            f'Try increasing DATE_WINDOW_DAYS or switching orbit direction.'
        )

    composite = collection.median().clip(aoi)
    # VH/VV ratio: roadmap Step 7 feature engineering
    vh_vv_ratio = composite.select('VH').subtract(composite.select('VV')).rename('VH_VV_ratio')
    composite   = composite.addBands(vh_vv_ratio)

    print(f'  Window : {start_date} to {end_date}')
    print(f'  Scenes : {n_images}')
    return composite, n_images


# Roadmap Step 2: load Sentinel-1 for T2 (used in coffee detection Step 5)
print(f'Loading Sentinel-1 T2 ({DATE_T2})...')
s1_T2, n_s1_T2 = load_sentinel1(AOI, DATE_T2)
print(f'S1 T2 scenes: {n_s1_T2}')

---
## Section 4: Spectral Index Computation

**Identical to crop health pipeline** -- same four indices (NDVI, EVI, NDWI, NDMI).

**Roadmap Step 7 alignment (feature engineering):**
- NDVI mean and amplitude across T1/T2 (roadmap: NDVI mean, std, amplitude)
- NDWI for moisture (roadmap: NDWI feature)
- EVI for dense vegetation (roadmap: EVI feature)

**Key adaptation:** Computed for BOTH T1 and T2.
NDVI difference (T2 - T1) is the primary change signal (roadmap Step 4 and Step 8):
- Large negative = significant vegetation loss = potential deforestation
- Near zero = stable land cover
- Positive = vegetation gain

**Note for Bhagwan:** The index arrays produced here feed directly into
your training dataset (roadmap Step -- Training Dataset Creation).

In [ ]:
def compute_indices(image):
    """
    computes vegetation and moisture indices from a Sentinel-2 image.
    all bands referenced by S2_SR_HARMONIZED naming convention.

    bands used:
        B2  = Blue  (10m) — EVI atmospheric correction
        B4  = Red   (10m) — NDVI, EVI
        B8  = NIR   (10m) — NDVI, EVI, NDWI, NDMI
        B11 = SWIR1 (20m) — NDWI (Gao 1996, canopy water)
        B12 = SWIR2 (20m) — NDMI (plant/soil moisture)

    returns: ee.Image with bands ['NDVI', 'EVI', 'NDWI', 'NDMI']
    """
    blue  = image.select('B2')
    red   = image.select('B4')
    nir   = image.select('B8')
    swir1 = image.select('B11')
    swir2 = image.select('B12')

    # NDVI: normalized difference vegetation index
    # high NDVI = dense healthy canopy; low NDVI = stressed or bare soil
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # EVI: enhanced vegetation index (Huete et al., 2002)
    # Ccrrects for canopy background and atmospheric effects
    # less prone to saturation in dense crops than NDVI
    evi = (nir.subtract(red)
              .multiply(2.5)
              .divide(
                  nir.add(red.multiply(6))
                     .subtract(blue.multiply(7.5))
                     .add(1)
              )
              .rename('EVI'))

    # NDWI (Gao 1996): normalized difference water index
    # uses NIR/SWIR1 —> sensitive to liquid water in vegetation canopy
    # note: NOT the McFeeters NDWI (Green/NIR), which detects open water bodies
    ndwi = nir.subtract(swir1).divide(nir.add(swir1)).rename('NDWI')

    # NDMI: normalized difference moisture index
    # uses NIR/SWIR2 — sensitive to plant and soil moisture stress
    ndmi = nir.subtract(swir2).divide(nir.add(swir2)).rename('NDMI')

    return ee.Image([ndvi, evi, ndwi, ndmi])


# Roadmap Step 7: compute indices for both time periods
indices_T1 = compute_indices(s2_T1)
indices_T2 = compute_indices(s2_T2)

# NDVI difference: roadmap Step 4 (forest loss) and Step 8 (time series change)
ndvi_diff_gee = indices_T2.select('NDVI').subtract(indices_T1.select('NDVI')).rename('NDVI_diff')

# download index arrays for local ML steps
# geemap.ee_to_numpy downloads a small AOI as a numpy array

def download_band(image, band_name, aoi, scale=20):
    """
    downloads a single-band GEE image to a numpy array.
    scale=20m matches SWIR band resolution (coarsest band used).
    """
    band = image.select(band_name)
    arr = geemap.ee_to_numpy(band, region=aoi, scale=scale)
    if arr is None:
        raise ValueError(f'Failed to download band {band_name}. Check AOI and date.')
    return arr[:, :, 0].astype(np.float32)

# Roadmap Step 7: all feature arrays at 30m (matches Hansen resolution)
SCALE = 30

# T1 arrays
ndvi_T1  = download_band(indices_T1, 'NDVI', AOI, SCALE)
evi_T1   = download_band(indices_T1, 'EVI',  AOI, SCALE)

# T2 arrays (full feature stack for coffee detection)
ndvi_T2  = download_band(indices_T2, 'NDVI', AOI, SCALE)
evi_T2   = download_band(indices_T2, 'EVI',  AOI, SCALE)
ndwi_T2  = download_band(indices_T2, 'NDWI', AOI, SCALE)
ndmi_T2  = download_band(indices_T2, 'NDMI', AOI, SCALE)

# Sentinel-1 radar features (roadmap Step 5 and Step 7)
vh_T2    = download_band(s1_T2, 'VH',          AOI, SCALE)
vv_T2    = download_band(s1_T2, 'VV',          AOI, SCALE)
vh_vv_T2 = download_band(s1_T2, 'VH_VV_ratio', AOI, SCALE)

# NDVI difference: roadmap Step 4 and Step 8
ndvi_diff_arr = ndvi_T2 - ndvi_T1

# Valid mask: all arrays must be finite
valid_mask = (
    np.isfinite(ndvi_T1)  & np.isfinite(ndvi_T2) &
    np.isfinite(evi_T2)   & np.isfinite(ndwi_T2)  &
    np.isfinite(ndmi_T2)  & np.isfinite(vh_T2)    &
    np.isfinite(vv_T2)
)

print(f'Array shape      : {ndvi_T1.shape}')
print(f'Valid pixels     : {valid_mask.sum()} / {valid_mask.size} ({100*valid_mask.mean():.1f}%)')
print(f'NDVI T1 range    : [{np.nanmin(ndvi_T1):.3f}, {np.nanmax(ndvi_T1):.3f}]')
print(f'NDVI T2 range    : [{np.nanmin(ndvi_T2):.3f}, {np.nanmax(ndvi_T2):.3f}]')
print(f'NDVI diff range  : [{np.nanmin(ndvi_diff_arr):.3f}, {np.nanmax(ndvi_diff_arr):.3f}]')
print(f'VH range (dB)    : [{np.nanmin(vh_T2):.1f}, {np.nanmax(vh_T2):.1f}]')

---
## Section 5: CHIRPS Rainfall Integration

**Same function as crop health pipeline.**

**Roadmap Step 2 alignment:** CHIRPS rainfall is explicitly listed as a required
data source for suitability context and irrigation monitoring.

**One change:** Lookback is now 365 days (annual rainfall) instead of 30 days.
For deforestation, annual rainfall determines whether the region supports coffee
growing (1000-3000mm range from roadmap suitability conditions).
This is suitability context, not short-term stress diagnosis.

In [ ]:
def get_rainfall_context(aoi, date_center, lookback_days=RAINFALL_LOOKBACK_DAYS):
    """
    retrieves mean cumulative precipitation (mm) over the AOI
    for the period [date_center - lookback_days, date_center].

    Data source: UCSB-CHG/CHIRPS/DAILY (0.05 degree resolution, daily)
    Available from 1981-01-01 to near-present.

    args:
        aoi: ee.Geometry
        date_center: str 'YYYY-MM-DD'
        lookback_days: int — agronomically relevant pre-analysis window

    returns:
        total_mm: float — cumulative mean precipitation over AOI
        daily_mm: float — mean daily precipitation
        rainfall_class: str — 'Low' / 'Moderate' / 'Adequate'
    """
    center_dt  = datetime.strptime(date_center, '%Y-%m-%d')
    start_date = (center_dt - timedelta(days=lookback_days)).strftime('%Y-%m-%d')

    chirps = (
        ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterBounds(aoi)
        .filterDate(start_date, date_center)
        .select('precipitation')
    )

    # sum daily precipitation across the window
    total_precip = chirps.sum().clip(aoi)

    # compute mean over AOI pixels
    mean_dict = total_precip.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=5000,   # CHIRPS native resolution ~5km
        maxPixels=1e6
    )

    total_mm = mean_dict.get('precipitation').getInfo()
    if total_mm is None:
        print('Warning: CHIRPS data unavailable for this region/date. Skipping rainfall context.')
        return None, None, 'Unknown'

    daily_mm = total_mm / lookback_days

    # classify rainfall (approximate thresholds for summer growing season)
    if total_mm < 20:
        rainfall_class = 'Low'
    elif total_mm < 60:
        rainfall_class = 'Moderate'
    else:
        rainfall_class = 'Adequate'

    print(f'CHIRPS Rainfall ({start_date} to {date_center}):')
    print(f'  Total  : {total_mm:.1f} mm over {lookback_days} days')
    print(f'  Daily  : {daily_mm:.2f} mm/day')
    print(f'  Class  : {rainfall_class}')

    return total_mm, daily_mm, rainfall_class


# Roadmap Step 2: annual rainfall for suitability context
rainfall_total, rainfall_daily, rainfall_class = get_rainfall_context(AOI, DATE_T2)

---
## Section 5b: Suitability Mask + Forest Baseline + Loss + Coffee Detection

**New vs crop health pipeline. Required by roadmap Steps 3, 4, 5, 6.**

Four steps in sequence:
1. Suitability mask (Step 3): elevation 600-2200m + coffee belt latitude
2. Forest baseline T1 (Step 3): Hansen tree cover > 50% = forest in 2017
3. Forest loss (Step 4): Hansen loss year 2017-2023 OR NDVI drop
4. Coffee detection T2 (Step 5): NDVI + EVI + Sentinel-1 VH in coffee range

Then combined into core change detection (Step 6):
Forest at T1 AND Loss detected AND Coffee at T2 = coffee-driven deforestation

In [ ]:
# Roadmap Step 3: Suitability mask
srtm      = ee.Image('USGS/SRTMGL1_003').select('elevation')
elev_mask = srtm.gte(100).And(srtm.lte(ELEVATION_MAX)).clip(AOI)
lat_mask  = ee.Image.pixelLonLat().select('latitude').abs().lte(COFFEE_BELT_LAT).clip(AOI)
suit_arr  = geemap.ee_to_numpy(elev_mask.And(lat_mask).toFloat(), region=AOI, scale=SCALE)
suitability_mask = suit_arr[:, :, 0].astype(bool) if suit_arr is not None else np.ones(valid_mask.shape, dtype=bool)

# Ensure suitability_mask has the same shape as valid_mask
if suitability_mask.shape != valid_mask.shape:
    print(f"Warning: Suitability mask shape {suitability_mask.shape} differs from valid_mask shape {valid_mask.shape}. Reshaping by cropping.")
    # Crop suitability_mask to match the dimensions of valid_mask
    suitability_mask = suitability_mask[:valid_mask.shape[0], :valid_mask.shape[1]]

print(f'Suitability mask: {suitability_mask.sum():,} pixels ({100*suitability_mask.mean():.1f}%)')

# Roadmap Step 3: Baseline forest mask T1
hansen     = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
tree_cover = hansen.select('treecover2000')
loss_year  = hansen.select('lossyear')
pre_loss   = loss_year.gt(0).And(loss_year.lt(17))
forest_arr = geemap.ee_to_numpy(tree_cover.gte(TREE_COVER_THRESHOLD).And(pre_loss.Not()).clip(AOI).toFloat(), region=AOI, scale=SCALE)
hansen_mask    = forest_arr[:, :, 0].astype(bool) if forest_arr is not None else np.zeros(valid_mask.shape, dtype=bool)
forest_mask_T1 = hansen_mask & (ndvi_T1 >= 0.5) & valid_mask
print(f'Forest at T1: {forest_mask_T1.sum():,} pixels ({100*forest_mask_T1.mean():.1f}%)')

# Roadmap Step 4: Forest loss detection
hansen_loss_arr  = geemap.ee_to_numpy(loss_year.gte(17).And(loss_year.lte(23)).clip(AOI).toFloat(), region=AOI, scale=SCALE)
hansen_loss_mask = hansen_loss_arr[:, :, 0].astype(bool) if hansen_loss_arr is not None else np.zeros(valid_mask.shape, dtype=bool)
ndvi_loss_mask   = (ndvi_diff_arr < NDVI_LOSS_THRESHOLD) & valid_mask
loss_mask        = (hansen_loss_mask | ndvi_loss_mask) & valid_mask
print(f'Forest loss: {loss_mask.sum():,} pixels (Hansen: {hansen_loss_mask.sum():,} | NDVI drop: {ndvi_loss_mask.sum():,})')

# Roadmap Step 5: Coffee detection at T2
ndvi_coffee    = (ndvi_T2 >= COFFEE_NDVI_MIN) & (ndvi_T2 <= COFFEE_NDVI_MAX)
evi_coffee     = (evi_T2  >= COFFEE_EVI_MIN)  & (evi_T2  <= COFFEE_EVI_MAX)
vh_coffee      = (vh_T2   >= COFFEE_VH_MIN)   & (vh_T2   <= COFFEE_VH_MAX)
coffee_mask_T2 = ndvi_coffee & evi_coffee & vh_coffee & valid_mask & suitability_mask
print(f'Coffee at T2: {coffee_mask_T2.sum():,} pixels')

# Roadmap Step 6: Three-way intersection
pixel_ha             = (SCALE ** 2) / 10000
coffee_deforestation = forest_mask_T1 & loss_mask & coffee_mask_T2
forest_loss_other    = forest_mask_T1 & loss_mask & ~coffee_mask_T2
stable_forest        = forest_mask_T1 & ~loss_mask
total_loss           = (forest_loss_other.sum() + coffee_deforestation.sum()) * pixel_ha
coffee_pct           = 100 * coffee_deforestation.sum() / max(forest_loss_other.sum() + coffee_deforestation.sum(), 1)

print(f'Roadmap Step 6 -- Change Detection:')
print(f'  Stable forest              : {stable_forest.sum() * pixel_ha:,.0f} ha')
print(f'  Forest loss (non-coffee)   : {forest_loss_other.sum() * pixel_ha:,.0f} ha')
print(f'  Coffee-driven deforestation: {coffee_deforestation.sum() * pixel_ha:,.0f} ha')
print(f'  % of loss driven by coffee : {coffee_pct:.1f}%')

# Build 4-class deforestation map
deforestation_map = np.zeros(valid_mask.shape, dtype=np.int8)
deforestation_map[stable_forest]                                        = 1
deforestation_map[forest_loss_other]                                    = 2
deforestation_map[~forest_mask_T1 & coffee_mask_T2 & valid_mask]       = 3
deforestation_map[coffee_deforestation]                                 = 4

---
## Section 6: Time Series Change Analysis

**Adapted from health classification in crop health pipeline.**

**Roadmap Step 8 alignment:** Track gradual conversion via NDVI change.
Classifies the NDVI difference (T2 - T1) into three change categories:
- Stable: NDVI change near zero -- land cover unchanged
- Moderate loss: NDVI dropped 0.15-0.35 -- partial vegetation loss or gradual conversion
- Significant loss: NDVI dropped > 0.35 -- sudden clearing event

Roadmap Step 8 notes: sudden drop = clearing, gradual stabilization = plantation growth.
This map supports that interpretation spatially.

In [ ]:
def classify_change(ndvi_diff, valid_mask=None):
    """
    Adapted from classify_health() in crop health pipeline.
    Roadmap Step 8: time series change analysis.

    Classifies NDVI difference into three change categories.
    Same structure as classify_health() -- quadrant-aware, area percentages.
    """
    change_map = np.zeros_like(ndvi_diff, dtype=np.int8)
    mask = valid_mask if valid_mask is not None else np.ones_like(ndvi_diff, dtype=bool)

    change_map[mask & (ndvi_diff >= NDVI_LOSS_THRESHOLD)]                                   = 1  # Stable
    change_map[mask & (ndvi_diff >= -0.35) & (ndvi_diff < NDVI_LOSS_THRESHOLD)]             = 2  # Moderate loss
    change_map[mask & (ndvi_diff < -0.35)]                                                   = 3  # Significant loss

    valid_pixels = mask.sum()
    pixel_ha = (SCALE ** 2) / 10000
    percentages = {
        'Stable'           : 100 * (change_map == 1).sum() / valid_pixels,
        'Moderate Loss'    : 100 * (change_map == 2).sum() / valid_pixels,
        'Significant Loss' : 100 * (change_map == 3).sum() / valid_pixels,
        'loss_ha'          : (change_map >= 2).sum() * pixel_ha,
    }

    # Quadrant analysis -- identical to crop health pipeline
    rows, cols = change_map.shape
    hmid, vmid = rows // 2, cols // 2
    quadrants  = {
        'NW': change_map[:hmid, :vmid],
        'NE': change_map[:hmid, vmid:],
        'SW': change_map[hmid:, :vmid],
        'SE': change_map[hmid:, vmid:],
    }
    class_labels = {1: 'Stable', 2: 'Moderate Loss', 3: 'Significant Loss'}
    quadrant_stats = {}
    for q_name, q_arr in quadrants.items():
        valid_q = q_arr[q_arr > 0]
        if len(valid_q) > 0:
            dominant  = int(np.bincount(valid_q).argmax())
            loss_pct  = 100 * (valid_q >= 2).sum() / len(valid_q)
            quadrant_stats[q_name] = {
                'dominant_class': class_labels.get(dominant, 'Unknown'),
                'loss_pct': loss_pct
            }

    print('NDVI Change Classification (Roadmap Step 8):')
    for cls, val in percentages.items():
        if cls == 'loss_ha':
            print(f'  Total loss area  : {val:.0f} ha')
        else:
            print(f'  {cls:20s}: {val:.1f}%')

    return change_map, percentages, quadrant_stats


change_map, change_pcts, change_quadrants = classify_change(
    ndvi_diff_arr, valid_mask=valid_mask
)


---
## Section 7: T2 Vegetation State

**Retained from crop health pipeline -- classify_health() unchanged.**

Applied to T2 NDVI/EVI to show current vegetation density in the AOI.
Helps interpret the change map:
- High vegetation at T2 = intact forest or recovered vegetation
- Moderate vegetation at T2 within loss zones = likely new plantation (coffee range)
- Low vegetation at T2 = recently cleared, bare soil

**Note for Bhagwan:** Moderate vegetation pixels at T2 within loss zones
are the primary candidate zones for your plantation detection model.

In [ ]:
t2_state_map, t2_state_pcts = classify_health(
    ndvi_T2, evi_T2, valid_mask=valid_mask
)

from matplotlib.colors import ListedColormap

display_map = t2_state_map.astype(float)
display_map[~valid_mask] = np.nan

cmap = ListedColormap(['#2d6a2d', '#f0c040', '#c0392b'])

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(display_map, cmap=cmap, vmin=1, vmax=3, interpolation='nearest')
ax.set_title(f'T2 Vegetation State Map\n{DATE_T2}', fontsize=13, fontweight='bold')
ax.axis('off')

legend_elements = [
    mpatches.Patch(color='#2d6a2d', label=f"High vegetation / forest ({t2_state_pcts['High vegetation (forest/dense)']:.1f}%)"),
    mpatches.Patch(color='#f0c040', label=f"Moderate vegetation / plantation candidate ({t2_state_pcts['Moderate vegetation (plantation candidate)']:.1f}%)"),
    mpatches.Patch(color='#c0392b', label=f"Low vegetation / bare or cleared ({t2_state_pcts['Low vegetation (bare/cleared)']:.1f}%)"),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9,
          framealpha=0.9, title='Vegetation State')

plt.tight_layout()
plt.show()

---
## Section 8: Hotspot Detection

**Adapted from crop health pipeline -- same Isolation Forest + DBSCAN.**

**Roadmap Step 7 alignment:** Feature vector now uses the full multi-sensor stack:
- NDVI_diff (change signal)
- NDVI_T1, NDVI_T2 (before and after)
- EVI_T2, NDWI_T2 (optical features)
- VH_T2, VH_VV_T2 (radar features -- roadmap Step 7)

Pixels anomalous across ALL signals simultaneously are the strongest
deforestation candidates. DBSCAN eps increased to 0.01 (from 0.001)
because deforestation patches are spatially larger than crop stress zones.

In [ ]:
def compute_uniformity(ndvi, valid_mask=None, n_clusters=N_CLUSTERS,
                       uniformity_bins=UNIFORMITY_BINS):
    """
    computes field uniformity using K-Means clustering on NDVI pixel values.

    the dominant cluster's share of total pixels defines the uniformity score.

    a perfectly uniform field would have all pixels in one cluster (score=100).

    returns:
        cluster_map: 2D array of cluster labels
        score: int 0–100
        label: str 'Uniform' / 'Mixed' / 'Patchy'
    """
    mask = valid_mask if valid_mask is not None else np.ones_like(ndvi, dtype=bool)

    flat_ndvi = ndvi[mask].reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    flat_labels = kmeans.fit_predict(flat_ndvi)

    # reconstruct 2D cluster map
    cluster_map = np.full(ndvi.shape, -1, dtype=np.int8)
    cluster_map[mask] = flat_labels

    # uniformity score: dominant cluster share
    counts = np.bincount(flat_labels)
    dominant_pct = counts.max() / counts.sum()
    score = round(dominant_pct * 100)

    u_thresh, m_thresh = uniformity_bins
    label = 'Uniform' if score >= u_thresh else 'Mixed' if score >= m_thresh else 'Patchy'

    print(f'Field Uniformity:')
    print(f'  Score  : {score}/100')
    print(f'  Label  : {label}')
    for i, c in enumerate(counts):
        print(f'  Cluster {i}: {100*c/counts.sum():.1f}% of field')

    return cluster_map, score, label


def detect_anomalies(ndvi, evi, ndwi, ndmi, valid_mask=None,
                     contamination=ISOLATION_CONTAMINATION):
    """
    multivariate anomaly detection using Isolation Forest.

    feature vector per pixel: [NDVI, EVI, NDWI, NDMI]
    flags pixels that are simultaneously anomalous across all four signals.
    --> stronger than thresholding any single index.

    returns:
        anomaly_mask: 2D boolean array (True = anomaly pixel)
        anomaly_scores: 2D float array (more negative = stronger anomaly)
    """
    mask = valid_mask if valid_mask is not None else np.ones_like(ndvi, dtype=bool)

    # Build feature matrix: (n_valid_pixels, 4)
    # Roadmap Step 7: multi-sensor feature vector
    # ndvi=ndvi_diff, evi=evi_T2, ndwi=vh_T2, ndmi=vh_vv_T2
    X = np.stack([
        ndvi[mask], evi[mask], ndwi[mask], ndmi[mask]
    ], axis=1)

    # Standardize features before Isolation Forest
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100
    )
    predictions = clf.fit_predict(X_scaled)   # -1 = anomaly, 1 = normal
    scores      = clf.decision_function(X_scaled)  # more negative = more anomalous

    # Reconstruct 2D arrays
    anomaly_mask   = np.zeros(ndvi.shape, dtype=bool)
    anomaly_scores = np.zeros(ndvi.shape, dtype=np.float32)
    anomaly_mask[mask]   = (predictions == -1)
    anomaly_scores[mask] = scores

    n_anomalies = anomaly_mask.sum()
    print(f'Isolation Forest:')
    print(f'  Anomaly pixels: {n_anomalies} ({100*n_anomalies/mask.sum():.1f}% of valid)')

    return anomaly_mask, anomaly_scores


def cluster_hotspots(anomaly_mask, anomaly_scores, n_hotspots=N_HOTSPOTS,
                     eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES):
    """
    spatially clusters anomaly pixels into coherent hotspot zones using DBSCAN.

    DBSCAN chosen over K-Means because:
    - Number of clusters is unknown in advance
    - Handles irregular field geometries naturally
    - Treats isolated single-pixel anomalies as noise (label=-1)

    returns:
        hotspot_clusters: list of dicts with cluster metadata
        cluster_label_map: 2D array of DBSCAN cluster labels
    """
    rows, cols = anomaly_mask.shape

    # get pixel coordinates of anomalies
    anomaly_rows, anomaly_cols = np.where(anomaly_mask)

    if len(anomaly_rows) < min_samples:
        print('Insufficient anomaly pixels for DBSCAN clustering.')
        return [], np.full(anomaly_mask.shape, -1)

    # normalize coordinates to [0,1] range before DBSCAN
    coords = np.stack([anomaly_rows / rows, anomaly_cols / cols], axis=1)

    db = DBSCAN(eps=eps, min_samples=min_samples)
    db_labels = db.fit_predict(coords)

    # reconstruct full-image cluster label map
    cluster_label_map = np.full(anomaly_mask.shape, -1, dtype=np.int16)
    cluster_label_map[anomaly_rows, anomaly_cols] = db_labels

    # summarize each cluster
    unique_labels = [l for l in np.unique(db_labels) if l >= 0]
    hotspot_clusters = []

    for label in unique_labels:
        cluster_pix_rows = anomaly_rows[db_labels == label]
        cluster_pix_cols = anomaly_cols[db_labels == label]
        cluster_scores   = anomaly_scores[cluster_pix_rows, cluster_pix_cols]

        # determine quadrant of cluster centroid
        centroid_row = cluster_pix_rows.mean() / rows
        centroid_col = cluster_pix_cols.mean() / cols
        v_pos = 'North' if centroid_row < 0.5 else 'South'
        h_pos = 'west'  if centroid_col < 0.5 else 'east'
        quadrant = f'{v_pos}{h_pos}'

        hotspot_clusters.append({
            'label'        : label,
            'n_pixels'     : len(cluster_pix_rows),
            'mean_score'   : cluster_scores.mean(),
            'min_score'    : cluster_scores.min(),  # most anomalous pixel
            'centroid_row' : centroid_row,
            'centroid_col' : centroid_col,
            'quadrant'     : quadrant,
            'pixel_rows'   : cluster_pix_rows,
            'pixel_cols'   : cluster_pix_cols,
        })

    # rank by severity (most anomalous score = lowest decision function value)
    hotspot_clusters.sort(key=lambda x: x['mean_score'])
    hotspot_clusters = hotspot_clusters[:n_hotspots]

    print(f'DBSCAN Hotspot Detection:')
    print(f'  Total clusters found : {len(unique_labels)}')
    print(f'  Top {n_hotspots} hotspots identified:')
    for i, hs in enumerate(hotspot_clusters):
        print(f'    Hotspot {i+1}: {hs["n_pixels"]} pixels, {hs["quadrant"]} region')

    return hotspot_clusters, cluster_label_map


# run uniformity and hotspot pipeline
# Uniformity on T2 NDVI -- shows current landscape structure
cluster_map, uniformity_score, uniformity_label = compute_uniformity(
    ndvi_T2, valid_mask=valid_mask
)

# Feature vector: NDVI_diff, EVI_T2, VH_T2, VH_VV_T2
# Roadmap Step 7: multi-sensor features for anomaly detection
anomaly_mask, anomaly_scores = detect_anomalies(
    ndvi_diff_arr, evi_T2, vh_T2, vh_vv_T2, valid_mask=valid_mask
)

hotspot_clusters, cluster_label_map = cluster_hotspots(
    anomaly_mask, anomaly_scores
)
print(f'Uniformity: {uniformity_label} ({uniformity_score}/100)')

---
## Section 9: Visualization

**Adapted from crop health pipeline.**

Maps:
1. True-color T1 (2017) -- forest baseline
2. True-color T2 (2023) -- post-conversion
3. NDVI difference map -- where vegetation changed (roadmap Step 8)
4. Coffee-driven deforestation map -- four-class output (roadmap Step 6)
5. Hotspot overlay -- anomalous deforestation zones (roadmap Step 10)

In [ ]:

def plot_deforestation_map(deforestation_map, valid_mask, coffee_ha, total_loss_ha):
    """
    Plots the 4-class coffee-driven deforestation map.
    Roadmap Step 6 output: Forest loss + Coffee detection combined.
    """
    from matplotlib.colors import ListedColormap
    display = deforestation_map.astype(float)
    display[~valid_mask] = np.nan

    cmap = ListedColormap(['#2d6a2d', '#f5a623', '#f0e040', '#c0392b'])
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(display, cmap=cmap, vmin=1, vmax=4, interpolation='nearest')
    ax.set_title(
        f'Coffee-Driven Deforestation Map (Roadmap Step 6)\n'
        f'Cacoal, Rondonia  {DATE_T1} to {DATE_T2}',
        fontsize=13, fontweight='bold'
    )
    ax.axis('off')
    legend_elements = [
        mpatches.Patch(color='#2d6a2d', label='Stable forest'),
        mpatches.Patch(color='#f5a623', label=f'Forest loss (non-coffee) ({total_loss_ha - coffee_ha:.0f} ha)'),
        mpatches.Patch(color='#f0e040', label='Coffee on non-forest land'),
        mpatches.Patch(color='#c0392b', label=f'Coffee-driven deforestation ({coffee_ha:.0f} ha)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9,
              framealpha=0.9, title='Land Cover Change')
    plt.tight_layout()
    plt.show()

def plot_classified_map(data_map, percentages, title, class_labels, class_colors, valid_mask=None):
    """
    Plots a generic N-class classification map with legend and area percentages.
    """
    display_map = data_map.astype(float)
    if valid_mask is not None:
        display_map[~valid_mask] = np.nan

    cmap = ListedColormap(class_colors)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(display_map, cmap=cmap, vmin=1, vmax=len(class_labels), interpolation='nearest')
    ax.set_title(f'{title}\n{DATE_CENTER}', fontsize=13, fontweight='bold')
    ax.axis('off')

    legend_elements = []
    for i, label_key in enumerate(class_labels):
        if label_key in percentages:
            legend_elements.append(
                mpatches.Patch(color=class_colors[i], label=f"{label_key} ({percentages[label_key]:.1f}%)")
            )
        else:
             legend_elements.append(
                mpatches.Patch(color=class_colors[i], label=f"{label_key}")
            )

    ax.legend(handles=legend_elements, loc='lower right', fontsize=10,
              framealpha=0.9, title='Classification')

    plt.tight_layout()
    plt.show()


def plot_water_map(water_map, percentages, valid_mask=None):
    """
    plots the 3-class water status map with legend and area percentages.
    """
    display_map = water_map.astype(float)
    if valid_mask is not None:
        display_map[~valid_mask] = np.nan

    cmap = ListedColormap(['#1a6faf', '#f5a623', '#b03a2e'])  # Blue / Orange / Red

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(display_map, cmap=cmap, vmin=1, vmax=3, interpolation='nearest')
    ax.set_title(f'Water Status Map\n{DATE_CENTER}', fontsize=13, fontweight='bold')
    ax.axis('off')

    legend_elements = [
        mpatches.Patch(color='#1a6faf', label=f"Adequate ({percentages['Adequate']:.1f}%)"),
        mpatches.Patch(color='#f5a623', label=f"Mild Deficit ({percentages['Mild Deficit']:.1f}%)"),
        mpatches.Patch(color='#b03a2e', label=f"Strong Deficit ({percentages['Strong Deficit']:.1f}%)"),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10,
              framealpha=0.9, title='Water Status')

    plt.tight_layout()
    plt.show()


def plot_uniformity_hotspots(ndvi, hotspot_clusters, anomaly_mask,
                             uniformity_score, uniformity_label, valid_mask=None):
    """
    plots NDVI continuous colormap with hotspot cluster bounding boxes overlaid.
    """
    display_ndvi = ndvi.copy()
    if valid_mask is not None:
        display_ndvi[~valid_mask] = np.nan

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(display_ndvi, cmap='RdYlGn', vmin=0, vmax=0.8, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='NDVI', fraction=0.046, pad=0.04)

    ax.set_title(
        f'Field Uniformity & Stress Hotspots\n'
        f'Uniformity: {uniformity_label} (score {uniformity_score}/100)',
        fontsize=12, fontweight='bold'
    )
    ax.axis('off')

    # overlay hotspot bounding boxes
    colors = ['#ff6b6b', '#ffa500', '#ffff00', '#ff69b4', '#ff4500']
    for i, hs in enumerate(hotspot_clusters):
        r_min, r_max = hs['pixel_rows'].min(), hs['pixel_rows'].max()
        c_min, c_max = hs['pixel_cols'].min(), hs['pixel_cols'].max()
        rect = plt.Rectangle(
            (c_min, r_min), c_max - c_min, r_max - r_min,
            linewidth=2, edgecolor=colors[i % len(colors)],
            facecolor='none', label=f'Hotspot {i+1} ({hs["quadrant"]})'
        )
        ax.add_patch(rect)
        ax.text(c_min + 2, r_min + 4, f'H{i+1}',
                color=colors[i % len(colors)], fontsize=9, fontweight='bold')

    if hotspot_clusters:
        ax.legend(loc='lower right', fontsize=9, framealpha=0.9, title='Stress Hotspots')

    plt.tight_layout()
    plt.show()


# render maps
def plot_true_color(composite, aoi, label='True Color (S2)'):
    """
    Displays a true-color Sentinel-2 composite via geemap.
    """
    m = geemap.Map()
    m.centerObject(aoi, zoom=11)
    m.addLayer(
        composite.select(['B4', 'B3', 'B2']),
        {'min': 0, 'max': 0.3, 'gamma': 1.4},
        label
    )
    m.add_basemap('SATELLITE')
    return m

# render maps
print('Map 1: True Color T1 (2017)')
tc_map = plot_true_color(s2_T1, AOI, label=f'True Color T1 ({DATE_T1})')
tc_map

In [ ]:
print('Map 2: True Color T2 (2023)')
plot_true_color(s2_T2, AOI, label=f'True Color T2 ({DATE_T2})')


In [ ]:
print('Map 3: NDVI Change Map (Roadmap Step 8)')

display_change = change_map.astype(float)
display_change[~valid_mask] = np.nan

cmap_change = ListedColormap(['#2d6a2d', '#f0c040', '#c0392b'])

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(display_change, cmap=cmap_change, vmin=1, vmax=3, interpolation='nearest')
ax.set_title(f'NDVI Change Classification Map\n{DATE_T1} to {DATE_T2}',
             fontsize=13, fontweight='bold')
ax.axis('off')

legend_elements = [
    mpatches.Patch(color='#2d6a2d', label=f"Stable ({change_pcts.get('Stable', 0):.1f}%)"),
    mpatches.Patch(color='#f0c040', label=f"Moderate Loss ({change_pcts.get('Moderate Loss', 0):.1f}%)"),
    mpatches.Patch(color='#c0392b', label=f"Significant Loss ({change_pcts.get('Significant Loss', 0):.1f}%)"),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10,
          framealpha=0.9, title='NDVI Change')

plt.tight_layout()
plt.show()

In [ ]:
print('Map 4: Coffee-Driven Deforestation (Roadmap Step 6)')
plot_deforestation_map(
    deforestation_map, valid_mask,
    coffee_ha=coffee_deforestation.sum() * pixel_ha,
    total_loss_ha=total_loss
)


# Section 9b: Summary Visualizations

In [ ]:

HEALTH_THRESHOLDS = [0.5, 0.3] # Define globally for plot_ndvi_histogram

def plot_class_breakdown(health_pcts, water_pcts):
    """
    side-by-side bar chart showing area percentage breakdown
    for vegetation health and water status classes.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Field Condition Summary: {DATE_CENTER}',
                 fontsize=14, fontweight='bold', y=1.02)

    # Health breakdown
    ax1 = axes[0]
    health_labels = list(health_pcts.keys())
    health_values = list(health_pcts.values())
    health_colors = ['#2d6a2d', '#f0c040', '#c0392b']

    bars = ax1.bar(health_labels, health_values, color=health_colors,
                   edgecolor='white', linewidth=1.2, width=0.5)

    for bar, val in zip(bars, health_values):
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.8,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')

    ax1.set_title('Vegetation Health', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Area (%)', fontsize=10)
    ax1.set_ylim(0, max(health_values) * 1.2)
    ax1.set_xticklabels(health_labels, fontsize=9)
    ax1.spines[['top', 'right']].set_visible(False)
    ax1.yaxis.grid(True, alpha=0.3)
    ax1.set_axisbelow(True)

    # water breakdown
    ax2 = axes[1]
    water_labels = list(water_pcts.keys())
    water_values = list(water_pcts.values())
    water_colors = ['#1a6faf', '#f5a623', '#b03a2e']

    bars2 = ax2.bar(water_labels, water_values, color=water_colors,
                    edgecolor='white', linewidth=1.2, width=0.5)

    for bar, val in zip(bars2, water_values):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.8,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')

    ax2.set_title('Water Status', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Area (%)', fontsize=10)
    ax2.set_ylim(0, max(water_values) * 1.2)
    ax2.set_xticklabels(water_labels, fontsize=9)
    ax2.spines[['top', 'right']].set_visible(False)
    ax2.yaxis.grid(True, alpha=0.3)
    ax2.set_axisbelow(True)

    plt.tight_layout()
    plt.show()


def plot_ndvi_histogram(ndvi, valid_mask=None, thresholds=HEALTH_THRESHOLDS):
    """
    NDVI pixel distribution with health classification threshold lines overlaid.
    visually justifies threshold choices by showing where natural breaks fall.
    """
    mask = valid_mask if valid_mask is not None else np.ones_like(ndvi, dtype=bool)
    flat_ndvi = ndvi[mask].flatten()
    flat_ndvi = flat_ndvi[np.isfinite(flat_ndvi)]

    h_thresh, m_thresh = thresholds

    fig, ax = plt.subplots(figsize=(9, 5))

    n, bins, patches = ax.hist(flat_ndvi, bins=60, edgecolor='none', alpha=0.85)

    # color bars by health class
    for patch, left_edge in zip(patches, bins[:-1]):
        if left_edge >= h_thresh:
            patch.set_facecolor('#2d6a2d')   # Healthy
        elif left_edge >= m_thresh:
            patch.set_facecolor('#f0c040')   # Moderate Stress
        else:
            patch.set_facecolor('#c0392b')   # High Stress

    # threshold lines
    ax.axvline(h_thresh, color='#2d6a2d', linestyle='--', linewidth=1.8,
               label=f'Healthy threshold ({h_thresh})')
    ax.axvline(m_thresh, color='#f0c040', linestyle='--', linewidth=1.8,
               label=f'Stress threshold ({m_thresh})')

    # annotate class regions
    ymax = n.max()
    ax.text((flat_ndvi.min() + m_thresh) / 2, ymax * 0.88,
            'High\nStress', color='#c0392b', ha='center', fontsize=9, fontweight='bold')
    ax.text((m_thresh + h_thresh) / 2, ymax * 0.88,
            'Moderate\nStress', color='#9a7d0a', ha='center', fontsize=9, fontweight='bold')
    ax.text((h_thresh + min(flat_ndvi.max(), 1.0)) / 2, ymax * 0.88,
            'Healthy', color='#2d6a2d', ha='center', fontsize=9, fontweight='bold')

    ax.set_xlabel('NDVI', fontsize=11)
    ax.set_ylabel('Pixel Count', fontsize=11)

    ax.set_title(f'NDVI Distribution with Health Classification Thresholds\n{DATE_T2}',
             fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

    # summary stats box
    stats_text = (f'n = {len(flat_ndvi):,} pixels\n'
                  f'mean NDVI = {flat_ndvi.mean():.3f}\n'
                  f'std = {flat_ndvi.std():.3f}')
    ax.text(0.02, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()


# render summary figures
# print('Figure 1: Class Breakdown')
# plot_class_breakdown(t2_state_pcts, water_pcts)

print('Figure 2: NDVI Distribution')
plot_ndvi_histogram(ndvi_T2, valid_mask=valid_mask)



---
## Section 10: Automated Report

**Adapted from crop health pipeline -- same quadrant-aware structure.**

Roadmap Step 10 alignment: outputs area of forest lost (hectares),
% converted to coffee, and region-wise statistics.

Annual CHIRPS rainfall used for coffee suitability context.

In [ ]:
def find_worst_quadrant(quadrant_stats, stress_key='loss_pct'):
    """
    Unchanged from crop health pipeline.
    Finds quadrant with highest vegetation loss.
    """
    if not quadrant_stats:
        return 'unknown'
    worst = max(quadrant_stats.items(), key=lambda x: x[1].get(stress_key, 0))
    names = {'NW': 'northwest', 'NE': 'northeast', 'SW': 'southwest', 'SE': 'southeast'}
    return names.get(worst[0], worst[0])


def generate_report(change_pcts, t2_state_pcts, uniformity_score, uniformity_label,
                    hotspot_clusters, change_quadrants, rainfall_total, rainfall_class,
                    coffee_ha, total_loss_ha, coffee_pct, date_T1, date_T2, n_T1, n_T2):
    """
    Adapted from crop health generate_report().
    Same quadrant-aware structure.
    Roadmap Step 10: area of forest lost, % converted to coffee, region stats.
    """
    worst_loss_q    = find_worst_quadrant(change_quadrants, stress_key='loss_pct')
    hotspot_summary = ', '.join([f'H{i+1}' for i in range(len(hotspot_clusters))]) \
                      if hotspot_clusters else 'not identified'

    rain_sentence = (
        f'Annual rainfall ({rainfall_total:.0f}mm) is within the 1000-3000mm coffee '
        f'suitability range, consistent with conditions supporting coffee expansion.'
        if rainfall_total and rainfall_total >= 1000
        else f'Annual rainfall ({rainfall_total:.0f}mm) is below optimal coffee range.'
    )

    report = f"""
=============================================================
 DEFORESTATION CHANGE DETECTION REPORT (Roadmap Step 10)
 Region : Cacoal, Rondonia, Brazil
 Period : {date_T1} to {date_T2}
 Data   : Sentinel-2 ({n_T1} T1, {n_T2} T2 scenes) + Sentinel-1 SAR
          + Hansen GFC + SRTM + CHIRPS
=============================================================

FOREST COVER CHANGE
  Total forest loss detected : {total_loss_ha:,.0f} ha
  Coffee-driven deforestation: {coffee_ha:,.0f} ha ({coffee_pct:.1f}% of total loss)
  Loss concentrated in       : {worst_loss_q} quadrant

NDVI CHANGE BREAKDOWN (Roadmap Step 8)
  Stable land cover  : {change_pcts.get('Stable', 0):.1f}%
  Moderate loss      : {change_pcts.get('Moderate Loss', 0):.1f}% (gradual conversion)
  Significant loss   : {change_pcts.get('Significant Loss', 0):.1f}% (sudden clearing)

CURRENT VEGETATION STATE T2
  High vegetation    : {t2_state_pcts.get('High vegetation (forest/dense)', 0):.1f}%
  Moderate vegetation: {t2_state_pcts.get('Moderate vegetation (plantation candidate)', 0):.1f}% (plantation candidate)
  Low vegetation     : {t2_state_pcts.get('Low vegetation (bare/cleared)', 0):.1f}%

RAINFALL SUITABILITY CONTEXT
  {rain_sentence}

HOTSPOT ZONES (Roadmap Step 10)
  {len(hotspot_clusters)} deforestation zone(s) detected (Isolation Forest + DBSCAN)
  Locations: {hotspot_summary}

LANDSCAPE UNIFORMITY
  Score: {uniformity_score}/100 -- {uniformity_label}

=============================================================
"""
    return report


report = generate_report(
    change_pcts, t2_state_pcts, uniformity_score, uniformity_label,
    hotspot_clusters, change_quadrants, rainfall_total, rainfall_class,
    coffee_ha=coffee_deforestation.sum() * pixel_ha,
    total_loss_ha=total_loss,
    coffee_pct=coffee_pct,
    date_T1=DATE_T1, date_T2=DATE_T2, n_T1=n_T1, n_T2=n_T2
)
print(report)


In [ ]:
# SRTM Slope and Aspect
# Roadmap Feature Engineering: terrain features (elevation, slope)

terrain = ee.Terrain.products(ee.Image('USGS/SRTMGL1_003'))

slope  = terrain.select('slope')
aspect = terrain.select('aspect')

# Trim to match valid_mask shape if GEE returns slightly different grid
slope_arr  = slope_arr[:valid_mask.shape[0], :valid_mask.shape[1]]
aspect_arr = aspect_arr[:valid_mask.shape[0], :valid_mask.shape[1]]

# Mask to valid pixels
slope_arr[~valid_mask]  = np.nan
aspect_arr[~valid_mask] = np.nan

print(f'Slope  range: [{np.nanmin(slope_arr):.1f}, {np.nanmax(slope_arr):.1f}] degrees')
print(f'Aspect range: [{np.nanmin(aspect_arr):.1f}, {np.nanmax(aspect_arr):.1f}] degrees')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(slope_arr, cmap='YlOrRd', interpolation='nearest')
axes[0].set_title('Slope (degrees)\nSRTM DEM', fontsize=12, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], label='Degrees', fraction=0.046, pad=0.04)

im2 = axes[1].imshow(aspect_arr, cmap='hsv', interpolation='nearest')
axes[1].set_title('Aspect (degrees)\nSRTM DEM', fontsize=12, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], label='Degrees', fraction=0.046, pad=0.04)

plt.suptitle(f'Terrain Features — Cacoal, Rondonia\n{DATE_T1} to {DATE_T2}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Slope and aspect arrays added to feature stack.')
print('Available for Bhagwan model: slope_arr, aspect_arr')

In [ ]:
# NDVI Time Series
# Roadmap Time-Series Change Analysis: monthly NDVI to detect
# sudden drop (deforestation) vs gradual stabilization (plantation growth)

def compute_monthly_ndvi(aoi, year, month, cloud_threshold=CLOUD_THRESHOLD):
    """
    Computes mean NDVI for a single month over the AOI.
    Returns mean NDVI value or NaN if no clean scenes available.
    """
    start = f'{year}-{month:02d}-01'
    # last day of month
    import calendar
    last_day = calendar.monthrange(year, month)[1]
    end = f'{year}-{month:02d}-{last_day}'

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
        .map(mask_s2_clouds)
    )

    n = collection.size().getInfo()
    if n == 0:
        return None, start

    composite = collection.median().clip(aoi)
    ndvi_img  = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

    mean_dict = ndvi_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=SCALE,
        maxPixels=1e6
    )
    mean_ndvi = mean_dict.get('NDVI').getInfo()
    return mean_ndvi, start


# Build time series: July each year 2017-2023
# July = dry season in Rondonia, consistent cloud-free window
print('Building NDVI time series (2017-2023, dry season)...')
print('This may take a few minutes -- one GEE call per year.\n')

years      = list(range(2017, 2024))
ndvi_means = []
dates      = []

for year in years:
    mean_val, date_str = compute_monthly_ndvi(AOI, year, month=7)
    ndvi_means.append(mean_val)
    dates.append(date_str)
    status = f'{mean_val:.3f}' if mean_val is not None else 'no data'
    print(f'  {year}-07: NDVI = {status}')

# Plot time series
fig, ax = plt.subplots(figsize=(10, 5))

valid_years = [y for y, v in zip(years, ndvi_means) if v is not None]
valid_ndvi  = [v for v in ndvi_means if v is not None]

ax.plot(valid_years, valid_ndvi, marker='o', linewidth=2.5,
        markersize=8, color='#0D9488', markerfacecolor='white',
        markeredgewidth=2.5)

# Annotate each point
for y, v in zip(valid_years, valid_ndvi):
    ax.annotate(f'{v:.3f}', (y, v), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, color='#1E293B')

# Mark T1 and T2
ax.axvline(2017, color='#2563EB', linestyle='--', linewidth=1.5,
           label='T1 (forest baseline)')
ax.axvline(2023, color='#C0392B', linestyle='--', linewidth=1.5,
           label='T2 (post-conversion)')

# Shade the transition zone
ax.axvspan(2017, 2023, alpha=0.06, color='#C0392B', label='Transition window')

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Mean NDVI', fontsize=11)
ax.set_title(f'NDVI Time Series — Cacoal, Rondonia (July, dry season)\n'
             f'Detecting deforestation trajectory 2017-2023',
             fontsize=12, fontweight='bold')
ax.set_xticks(years)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# Interpret the trend
if valid_ndvi[-1] < valid_ndvi[0]:
    drop = valid_ndvi[0] - valid_ndvi[-1]
    print(f'\nNDVI dropped {drop:.3f} from {valid_ndvi[0]:.3f} (2017) to {valid_ndvi[-1]:.3f} (2023)')
    print('Roadmap interpretation: sustained NDVI decline consistent with forest loss and plantation establishment.')
else:
    print(f'\nNDVI relatively stable: {valid_ndvi[0]:.3f} (2017) to {valid_ndvi[-1]:.3f} (2023)')